# Home Credit Data Analysis

Exploratory data analysis notebook for the raw Home Credit dataset in `dataset/`.

This notebook focuses on:
- table sizes and dataset overview
- target distribution
- missingness in `application_train`
- key numeric feature behavior vs target
- simple correlation screening
- quick join coverage checks for the auxiliary tables

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "dataset").exists() and (ROOT.parent / "dataset").exists():
    ROOT = ROOT.parent
if not (ROOT / "dataset").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATASET_DIR = ROOT / "dataset"
ROOT

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

plt.style.use("ggplot")

In [ ]:
TABLES = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "pos_cash_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
}

overview_rows = []
for table_name, filename in TABLES.items():
    path = DATASET_DIR / filename
    preview = pd.read_csv(path, nrows=5)
    row_count = sum(1 for _ in path.open("r", encoding="utf-8")) - 1
    overview_rows.append(
        {
            "table": table_name,
            "rows": row_count,
            "columns": preview.shape[1],
            "path": str(path.relative_to(ROOT)),
        }
    )

overview = pd.DataFrame(overview_rows).sort_values("rows", ascending=False)
overview

In [ ]:
application_train = pd.read_csv(DATASET_DIR / "application_train.csv")
application_test = pd.read_csv(DATASET_DIR / "application_test.csv")

print("application_train shape:", application_train.shape)
print("application_test shape:", application_test.shape)
application_train.head()

In [ ]:
target_summary = pd.DataFrame(
    {
        "count": application_train["TARGET"].value_counts().sort_index(),
        "share": application_train["TARGET"].value_counts(normalize=True).sort_index(),
    }
)
target_summary.index.name = "TARGET"
target_summary

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
application_train["TARGET"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4C78A8", "#E45756"])
ax.set_title("Target Distribution")
ax.set_xlabel("TARGET")
ax.set_ylabel("Count")
plt.show()

In [ ]:
missing = (
    application_train.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing.head(25)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top_missing = missing.head(20).iloc[::-1]
ax.barh(top_missing["feature"], top_missing["missing_fraction"], color="#72B7B2")
ax.set_title("Top 20 Missing Features in application_train")
ax.set_xlabel("Missing Fraction")
plt.tight_layout()
plt.show()

In [ ]:
key_numeric_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

application_train[key_numeric_features + ["TARGET"]].describe().T

In [ ]:
plot_features = ["EXT_SOURCE_2", "EXT_SOURCE_3", "AMT_CREDIT", "AMT_ANNUITY"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.ravel(), plot_features, strict=True):
    good = application_train.loc[application_train["TARGET"] == 0, feature].dropna()
    bad = application_train.loc[application_train["TARGET"] == 1, feature].dropna()
    ax.hist(good, bins=40, alpha=0.6, density=True, label="TARGET=0", color="#4C78A8")
    ax.hist(bad, bins=40, alpha=0.6, density=True, label="TARGET=1", color="#E45756")
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = [
    col for col in application_train.columns
    if pd.api.types.is_numeric_dtype(application_train[col]) and col not in {"SK_ID_CURR", "TARGET"}
]

correlation_rows = []
target_values = application_train["TARGET"].to_numpy()
for col in numeric_columns:
    series = application_train[col]
    if series.nunique(dropna=True) < 20:
        continue
    filled = series.fillna(series.median())
    corr = np.corrcoef(filled, target_values)[0, 1]
    if np.isnan(corr):
        continue
    correlation_rows.append({"feature": col, "target_correlation": corr, "abs_correlation": abs(corr)})

correlations = pd.DataFrame(correlation_rows).sort_values("abs_correlation", ascending=False)
correlations.head(20)

In [ ]:
categorical_columns = [
    col for col in application_train.columns
    if application_train[col].dtype == "object"
]

cat_rate_rows = []
for col in categorical_columns:
    top_levels = application_train[col].fillna("<MISSING>").value_counts().head(5).index
    subset = application_train[application_train[col].fillna("<MISSING>").isin(top_levels)].copy()
    grouped = subset.assign(level=subset[col].fillna("<MISSING>")).groupby("level")["TARGET"].agg(["mean", "count"])
    for level, row in grouped.iterrows():
        cat_rate_rows.append(
            {
                "feature": col,
                "level": level,
                "default_rate": row["mean"],
                "count": row["count"],
            }
        )

categorical_default_rates = pd.DataFrame(cat_rate_rows)
categorical_default_rates.sort_values(["default_rate", "count"], ascending=[False, False]).head(20)

In [ ]:
join_tables = {
    "bureau": ("bureau.csv", "SK_ID_CURR"),
    "previous_application": ("previous_application.csv", "SK_ID_CURR"),
    "pos_cash_balance": ("POS_CASH_balance.csv", "SK_ID_CURR"),
    "credit_card_balance": ("credit_card_balance.csv", "SK_ID_CURR"),
    "installments_payments": ("installments_payments.csv", "SK_ID_CURR"),
}

coverage_rows = []
app_ids = set(application_train["SK_ID_CURR"])
for table_name, (filename, key) in join_tables.items():
    ids = pd.read_csv(DATASET_DIR / filename, usecols=[key])[key].dropna().unique()
    matched = sum(1 for value in ids if value in app_ids)
    coverage_rows.append(
        {
            "table": table_name,
            "unique_ids": len(ids),
            "matched_train_ids": matched,
            "share_of_train_ids": matched / application_train["SK_ID_CURR"].nunique(),
        }
    )

coverage = pd.DataFrame(coverage_rows).sort_values("share_of_train_ids", ascending=False)
coverage

In [ ]:
summary = {
    "target_default_rate": float(application_train["TARGET"].mean()),
    "top_missing_features": missing.head(10).to_dict(orient="records"),
    "top_target_correlations": correlations.head(10).to_dict(orient="records"),
    "join_coverage": coverage.to_dict(orient="records"),
}

summary